# Session 4 – Preparation: Pandas II & Matplotlib

## Setup

pandas builds on top of **NumPy**, and for the plots we add **Matplotlib**. You need to install them **once** in your terminal:

```bash
pip install numpy pandas matplotlib
```

Run the cell below and check that everything works:

In [ ]:
%pip install numpy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Why Pandas II?

Remember Session 3? You can now **select, filter and summarize** – but the data itself stayed untouched.

Real data analysis is a **workflow**: clean the data → transform it → aggregate it → **show** it. That's exactly this session. 

Let's take a look:

In [ ]:
import pandas as pd

orders = [
    {'item': 'coffee', 'cups': 12},
    {'item': 'cake',   'cups': 5},
    {'item': 'coffee', 'cups': 9},
    {'item': 'tea',    'cups': 7},
    {'item': 'cake',   'cups': 3},
    {'item': 'coffee', 'cups': 14},
]

sales = pd.DataFrame(orders)

# Collect the totals by hand
cups_per_item = {}
for order in orders:
    cups_per_item[order['item']] = cups_per_item.get(order['item'], 0) + order['cups']
print(cups_per_item)   # {'coffee': 35, 'cake': 8, 'tea': 7}

print()
# Collect the totals using Pandas: group, sum, sort – no loop
print(sales.groupby('item')['cups'].sum().sort_values(ascending=False))
# -> coffee 35, cake 8, tea 7

## 1. Cleaning Data

Real data is almost never clean: missing values, `-` placeholders, duplicate rows. Session 3 was about **spotting** such problems. Today you learn to **fix** them.

First we create a small practice dataset, which has the same classic problems:

- one order logged **twice**
- one **drink size** missing 
- a `-` placeholder in `payment`

⚠️ Note: food (cake, cookie) **never** has a size – an empty `size` for food is not an error.

In [ ]:
csv_text = '''order_id,date,weekday,time,customer,item,size,price,payment
1001,2026-10-05,Mon,08:15,Alice,coffee,large,3.10,card
1002,2026-10-05,Mon,08:30,Bob,espresso,small,2.90,cash
1003,2026-10-05,Mon,09:01,Charlie,tea,medium,2.70,card
1004,2026-10-05,Mon,09:05,Alice,cake,,3.90,card
1005,2026-10-06,Tue,08:12,Dave,coffee,large,3.10,card
1006,2026-10-06,Tue,08:30,Eve,cookie,,1.90,cash
1007,2026-10-06,Tue,10:15,Frank,hot chocolate,medium,3.50,cash
1008,2026-10-07,Wed,08:05,Grace,coffee,,3.10,card
1009,2026-10-07,Wed,08:45,Alice,tea,small,2.70,-
1010,2026-10-07,Wed,09:20,Bob,cake,,3.90,card
1011,2026-10-08,Thu,08:00,Charlie,coffee,large,3.10,card
1012,2026-10-08,Thu,08:22,Dave,espresso,small,2.90,cash
1012,2026-10-08,Thu,08:22,Dave,espresso,small,2.90,cash
1013,2026-10-08,Thu,09:40,Eve,cake,,3.90,card
1014,2026-10-09,Fri,08:10,Frank,coffee,medium,3.10,card
1015,2026-10-09,Fri,08:35,Grace,tea,large,2.70,cash
1016,2026-10-09,Fri,09:00,Alice,cookie,,1.90,card
1017,2026-10-10,Sat,09:15,Bob,hot chocolate,large,3.50,cash
1018,2026-10-10,Sat,09:45,Charlie,cake,,3.90,card
'''

with open('cafe_sales_week.csv', 'w') as f:
    f.write(csv_text)

print('cafe_sales_week.csv created')

In [ ]:
sales = pd.read_csv('cafe_sales_week.csv')

print(sales.shape)                 # (19, 9) -> 18 orders, one logged twice
print(sales.head())
print()
print(sales.isnull().sum())        # size: 7, payment: 0 (!)
print()
print('Duplicate rows:', sales.duplicated().sum())   # 1

# Fix 1: remove the duplicate
sales = sales.drop_duplicates()
print(sales.shape)                 # (18, 9)

## 2. Handling Missing Data

Now we handle the gaps. Two directions:

- `dropna()` – remove rows (or columns) with missing values
- `fillna()` – fill them with a value instead

⚠️ `dropna()` without arguments removes **every** row with any gap. With `subset=[...]` you drop only where a **specific** column is missing.

And: the `-` placeholder is not `NaN` yet – `replace()` makes it visible first.

In [ ]:
import numpy as np

# The '-' placeholder is NOT NaN yet -> make it visible first
sales = sales.replace('-', np.nan)
print(sales.isnull().sum())        # size: 7, payment: 1
print()

# dropna() removes every row with ANY gap
print(sales.dropna().shape)        # (10, 9) -> we would lose 8 of 18 orders!

# Targeted alternative: sales = sales.dropna(subset=['payment'])
# ...but order 1009 is valuable -> fill instead (with the most common value)
sales['payment'] = sales['payment'].fillna('card')

# Handling the size gaps: 6 are structural (food), 1 drink size is really lost
sales['size'] = sales['size'].fillna('unknown')

print(sales.isnull().sum().sum())  # 0 -> clean!
print(sales.shape)                 # (18, 9)

## 3. Type Conversions

- After `read_csv`, text columns have the dtype `object` – just strings. A real date column needs the right **type** first
- `pd.to_datetime()` converts a date column → dtype `datetime64`
- `astype()` for explicit conversions – `'category'` is perfect for columns with few repeated values

Why bother? A datetime column can be sorted correctly, grouped by month – and **checked**.

In [ ]:
print(sales['date'].dtype)         # object -> just strings so far

# Convert to real datetimes
sales['date'] = pd.to_datetime(sales['date'], format='%Y-%m-%d')
print(sales['date'].dtype)        # datetime64[ns]
print()

# Check: does the 'weekday' column match the real dates?
print((sales['weekday'] == sales['date'].dt.strftime('%a')).all())   # True
print()

# Explicit conversion: 'payment' has only two values -> category
sales['payment'] = sales['payment'].astype('category')
print(sales.dtypes)

## 4. Creating New Columns

The columns you need for an analysis rarely exist yet – you create them from the ones you have:

- Direct assignment: `sales['new'] = ...`
- With a condition: `np.where(condition, 'then', 'else')`
- With a list: `.isin([...])` returns True/False

In [ ]:
drinks = ['coffee', 'espresso', 'tea', 'hot chocolate']

sales['is_drink'] = sales['item'].isin(drinks)                          # True/False
sales['price_tier'] = np.where(sales['price'] >= 3.00, 'premium', 'standard')

print(sales[['item', 'price', 'is_drink', 'price_tier']].head())
print()
print(sales['is_drink'].sum(), 'drinks')          # 12
print(sales['price_tier'].value_counts())         # premium 11, standard 7

## 5. Sorting & Indexing

- `sort_values('price')` – by one column; `ascending=False` flips the order
- Several columns at once: `sort_values(['date', 'time'])`
- `set_index('order_id')` – give the rows meaningful labels
- `sort_index()` / `reset_index()`

In [ ]:
# The most expensive orders first
print(sales.sort_values('price', ascending=False).head(3))
print()

# Chronological: date first, then time
print(sales.sort_values(['date', 'time']).head(3))
print()

# order_id as row label – .loc[1005] now fetches one order directly
sales = sales.set_index('order_id')
print(sales.loc[1005])             # Dave's Tuesday coffee

# Back to the default index any time
# sales = sales.reset_index()

## 6. GroupBy & Aggregations

**Split – apply – combine**: `groupby` splits the table into groups (one per item), computes a statistic per group, and combines the results:

- `sales.groupby('item')['price'].mean()` – the average price per item
- `.agg([...])` – several statistics at once
- `.size()` – how many rows each group has (like `value_counts`)

This is the tool behind almost every *"per ..."* question: per item, per day, per customer.

In [ ]:
# Number of orders and average price per item
print(sales.groupby('item')['price'].agg(['count', 'mean']))
# -> cake 4 / 3.90, coffee 5 / 3.10, cookie 2 / 1.90,
#    espresso 2 / 2.90, hot chocolate 2 / 3.50, tea 3 / 2.70
print()

# Revenue per item, biggest first
print(sales.groupby('item')['price'].sum().round(2).sort_values(ascending=False))
# -> cake 15.60, coffee 15.50, tea 8.10, hot chocolate 7.00, espresso 5.80, cookie 3.80
print()

# How the guests paid
print(sales['payment'].value_counts())   # card 12, cash 6

## 7. Built-in Plotting

pandas Series and DataFrames can draw themselves: `.plot()` hands the data to Matplotlib behind the scenes.

- `kind='bar'` – bars, `kind='line'` – lines, `kind='hist'` – distributions
- `.plot()` on a groupby result = the fastest way to create a chart

Perfect for quick looks **during** the analysis – for polished charts we switch to Matplotlib itself in a moment.

In [ ]:
# Orders per item as a bar chart
sales['item'].value_counts().plot(kind='bar')

# Revenue per day as a line chart
sales.groupby('date')['price'].sum().plot(kind='line')

# The price distribution
sales['price'].plot(kind='hist', bins=6)

## 8. Matplotlib Basics

Matplotlib is the plotting library behind pandas:

- `import matplotlib.pyplot as plt` – the standard import
- The clean pattern: `fig, ax = plt.subplots()` – a figure (the canvas) and axes (one chart)
- `ax.plot(x, y)` draws into it, `plt.show()` displays it
- `figsize=(width, height)` – the size in inches

In [ ]:
import matplotlib.pyplot as plt

# Revenue per day – the data for our line chart
daily = sales.groupby('date')['price'].sum().round(2)
print(daily)
# -> 2026-10-05: 12.60, 2026-10-06: 8.50, 2026-10-07: 9.70,
#    2026-10-08: 9.90, 2026-10-09: 7.70, 2026-10-10: 7.40

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(daily.index, daily.values, marker='o')
plt.show()

## 9. Labels, Titles & Layout

A chart without labels is a riddle. The essentials:

- `ax.set_title()` / `set_xlabel()` / `set_ylabel()`
- `label=...` + `ax.legend()` when several series share one chart
- `ax.grid(True)` – guide lines for the eye
- `fig.tight_layout()` – no cut-off labels

With this, the daily revenue becomes a presentable chart:

In [ ]:
orders_per_day = sales.groupby('date').size()

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(daily.index, daily.values, marker='o', label='Revenue (EUR)')
ax.plot(orders_per_day.index, orders_per_day.values, marker='s', label='Orders')
ax.set_title('Café Python – Weekly Revenue and Orders')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue (EUR) / Orders')
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()

## 10. Plot Types

The four chart types you will need most:

- **Bar** – comparisons between categories (items)
- **Line** – developments over time (days)
- **Scatter** – relationships between two numbers (orders vs revenue)
- **Pie** – proportions of a whole (payment methods)

Best of all: a 2×2 dashboard with `plt.subplots(2, 2)` – four charts, one figure.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
(ax1, ax2), (ax3, ax4) = axes

# Bar: orders per item
sales['item'].value_counts().plot(kind='bar', ax=ax1, title='Orders per item')

# Line: revenue per day
daily.plot(kind='line', ax=ax2, title='Revenue per day')

# Scatter: more orders, more revenue?
ax3.scatter(orders_per_day.values, daily.values)
ax3.set_title('Orders vs revenue')
ax3.set_xlabel('Orders per day')
ax3.set_ylabel('Revenue (EUR)')

# Pie: how the guests paid
sales['payment'].value_counts().plot(kind='pie', ax=ax4, title='Payment', autopct='%1.0f%%')

fig.tight_layout()
plt.show()